# Assignment 2: Pandas — Messy Data Handling

**Dataset Source**: Netflix Movies and TV Shows (Public Dataset via GitHub / Kaggle)

**Source URL**: `https://raw.githubusercontent.com/prasertcbs/basic-dataset/master/netflix_titles.csv`

> **Dataset Characteristics**: 8,807 rows and 12 columns, containing mixed text and numeric features, categorical columns, date variables, real missing values, inconsistent categories, and duration text.

### 1. Load Data with Explicit Dtypes & Downcasting

In [1]:
import pandas as pd
import numpy as np
import time

url = "https://raw.githubusercontent.com/prasertcbs/basic-dataset/master/netflix_titles.csv"

# Load file with explicit dtype and parse_dates arguments
df = pd.read_csv(
    url,
    dtype={
        "show_id": str,
        "type": str,
        "title": str,
        "director": str,
        "cast": str,
        "country": str,
        "release_year": "int64",
        "rating": str,
        "duration": str,
        "listed_in": str,
        "description": str
    }
)

mem_before = df.memory_usage(deep=True).sum()
print(f"Initial Memory Usage: {mem_before:,} bytes ({mem_before / (1024**2):.2f} MB)")

# Downcast numeric columns
df["release_year"] = pd.to_numeric(df["release_year"], downcast="integer")

# Convert low-cardinality text columns to category
low_card_cols = ["type", "rating"]
for col in low_card_cols:
    df[col] = df[col].astype("category")

mem_after = df.memory_usage(deep=True).sum()
print(f"Memory Usage After Downcasting: {mem_after:,} bytes ({mem_after / (1024**2):.2f} MB)")

saving_pct = ((mem_before - mem_after) / mem_before) * 100
print(f"Memory Saving: {saving_pct:.2f}%")

Initial Memory Usage: 5,759,921 bytes (5.49 MB)
Memory Usage After Downcasting: 5,062,923 bytes (4.83 MB)
Memory Saving: 12.10%


### 2. Single Profiling DataFrame

In [2]:
# Profiling DataFrame: one row per column
profiling = pd.DataFrame({
    "dtype": df.dtypes,
    "missing count": df.isna().sum(),
    "missing percent": (df.isna().mean() * 100).round(2),
    "unique count": df.nunique(),
    "most frequent value": df.apply(lambda col: col.mode().iloc[0] if not col.mode().empty else np.nan),
    "frequency": df.apply(lambda col: col.value_counts().iloc[0] if not col.dropna().empty else 0)
})

profiling

,dtype,missing count,missing percent,unique count,most frequent value,frequency
show_id,object,0,0.00,6234,1005494,1
type,category,0,0.00,2,Movie,4265
title,object,0,0.00,6172,Limitless,3
director,object,1969,31.58,3301,"Raúl Campos, Jan Suter",18
cast,object,570,9.14,5469,David Attenborough,18
country,object,476,7.64,554,United States,2032
date_added,object,11,0.18,1524,"January 1, 2020",122
release_year,int16,0,0.00,72,2018,1063
rating,category,10,0.16,14,TV-MA,2027
duration,object,0,0.00,201,1 Season,1321


### 3. Duplicate Detection & Key Identification

In [3]:
exact_dupes = df.duplicated().sum()
print(f"Exact Duplicate Rows: {exact_dupes}")

# Key columns that should uniquely identify a title
key_cols = ["title", "type", "release_year"]
key_dupes = df.duplicated(subset=key_cols).sum()
print(f"Key Duplicates on {key_cols}: {key_dupes}")

# Rule: Drop subsequent duplicate occurrences, keeping the 'first' verified entry
df = df.drop_duplicates(subset=key_cols, keep="first").reset_index(drop=True)
print(f"Remaining rows after dropping key duplicates: {len(df):,}")

Exact Duplicate Rows: 0
Key Duplicates on ['title', 'type', 'release_year']: 3
Remaining rows after dropping key duplicates: 6,231


**Deduplication Rule Rationale**: We kept the `first` record because catalog scrapers append updates chronologically; preserving the initial entry retains the original indexing while eliminating redundant duplicate feeds.

### 4. Clean One Text Column (Inconsistent Labels)

In [4]:
print("Value counts BEFORE cleaning:")
print(df["rating"].value_counts().head(10))

# Clean text column 'rating': strip whitespace, standardize casing, and collapse synonyms
cleaned_rating = df["rating"].astype(str).str.strip().str.upper()
cleaned_rating = cleaned_rating.replace({
    "UR": "NR",           # 'Unrated' is standardized to 'Not Rated'
    "NAN": "UNRATED",
    "NONE": "UNRATED"
})
df["rating"] = cleaned_rating.astype("category")

print("\nValue counts AFTER cleaning:")
print(df["rating"].value_counts().head(10))

Value counts BEFORE cleaning:
rating
TV-MA    2025
TV-14    1697
TV-PG     701
R         508
PG-13     286
NR        218
PG        184
TV-Y7     169
TV-G      149
TV-Y      143
Name: count, dtype: int64

Value counts AFTER cleaning:
rating
TV-MA    2025
TV-14    1697
TV-PG     701
R         508
PG-13     286
NR        225
PG        184
TV-Y7     169
TV-G      149
TV-Y      143
Name: count, dtype: int64


### 5. Regex Number Extraction

In [5]:
# Extract numeric value from text column 'duration' (e.g., '90 min' -> 90, '2 Seasons' -> 2)
extracted_num = df["duration"].astype(str).str.extract(r"(\d+)", expand=False)
df["duration_num"] = pd.to_numeric(extracted_num, errors="coerce")

unmatched_rows = df["duration_num"].isna().sum()
print(f"Rows where regex extraction did not match: {unmatched_rows}")
print("\nSample of extracted numeric duration:")
df[["title", "duration", "duration_num"]].head(5)

Rows where regex extraction did not match: 0

Sample of extracted numeric duration:


,title,duration,duration_num
0,Norm of the North: King Sized Adventure,90 min,90
1,Jandino: Whatever it Takes,94 min,94
2,Transformers Prime,1 Season,1
3,Transformers: Robots in Disguise,1 Season,1
4,#realityhigh,99 min,99


### 6. Parse Date Column & Extract Temporal Features

In [6]:
# Parse date column with errors='coerce'
parsed_dates = pd.to_datetime(df["date_added"].astype(str).str.strip(), errors="coerce")
failed_dates = parsed_dates.isna().sum()
print(f"Date parsing failed / missing values count: {failed_dates}")

# Extract temporal columns
df["year_added"] = parsed_dates.dt.year
df["month_added"] = parsed_dates.dt.month
df["day_of_week"] = parsed_dates.dt.day_name()
df["quarter_added"] = parsed_dates.dt.quarter

df[["date_added", "year_added", "month_added", "day_of_week", "quarter_added"]].head(5)

Date parsing failed / missing values count: 11


,date_added,year_added,month_added,day_of_week,quarter_added
0,"September 9, 2019",2019.0,9.0,Monday,3.0
1,"September 9, 2016",2016.0,9.0,Friday,3.0
2,"September 8, 2018",2018.0,9.0,Saturday,3.0
3,"September 8, 2018",2018.0,9.0,Saturday,3.0
4,"September 8, 2017",2017.0,9.0,Friday,3.0


### 7. GroupBy with Named Aggregation (Including Custom Function)

In [7]:
# Custom aggregation function: calculates difference between max and min release years
def year_spread(s):
    return s.max() - s.min()

# Groupby with named aggregation carrying 4 different aggregations
summary_table = df.groupby("type", observed=True).agg(
    total_titles=("show_id", "count"),
    avg_duration=("duration_num", "mean"),
    median_release_year=("release_year", "median"),
    release_year_range=("release_year", year_spread)
)

summary_table

,total_titles,avg_duration,median_release_year,release_year_range
type,,,,
Movie,4263,99.086324,2016.0,78
TV Show,1968,1.779472,2017.0,95


### 8. Group Mean Deviation Using `transform`

In [8]:
# Add column holding each row's value minus group mean using transform
df["duration_diff_mean"] = df["duration_num"] - df.groupby("type", observed=True)["duration_num"].transform("mean")

df[["type", "title", "duration_num", "duration_diff_mean"]].head(5)

,type,title,duration_num,duration_diff_mean
0,Movie,Norm of the North: King Sized Adventure,90,-9.086324
1,Movie,Jandino: Whatever it Takes,94,-5.086324
2,TV Show,Transformers Prime,1,-0.779472
3,TV Show,Transformers: Robots in Disguise,1,-0.779472
4,Movie,#realityhigh,99,-0.086324


**Why `transform` is the right tool here**:
`transform` evaluates the aggregation per group and broadcasts the results directly back to the original DataFrame's index. This preserves the original row count and alignment without requiring an explicit join/merge operation that risks row order mutation.

### 9. Pivot Table and Lossless Round-Trip Melt

In [9]:
# Select top ratings to ensure complete contingency
top_ratings = ["TV-MA", "TV-14", "R", "PG-13", "TV-PG"]
pivot_sub = df[df["rating"].isin(top_ratings)].copy()

# 1. Pivot Table
pivot = pivot_sub.pivot_table(
    index="rating",
    columns="type",
    values="duration_num",
    aggfunc="mean",
    observed=True,
    fill_value=0
)
print("Pivot Table:")
print(pivot)

# 2. Melt back to long form
melted = pivot.reset_index().melt(
    id_vars="rating",
    var_name="type",
    value_name="duration_num"
)
print("\nMelted Form (sample):")
print(melted.head(4))

# 3. Re-pivot melted form to prove lossless round trip
re_pivoted = melted.pivot_table(
    index="rating",
    columns="type",
    values="duration_num",
    observed=True
)
assert np.allclose(pivot.values, re_pivoted.values), "Round trip failed!"
print("\nProof: np.allclose(pivot.values, re_pivoted.values) == True (Lossless round trip verified)")

Pivot Table:
type         Movie   TV Show
rating                      
PG-13   108.853147  0.000000
R       106.158103  1.000000
TV-14   108.137765  1.830046
TV-MA    95.243685  1.622975
TV-PG    97.949074  1.784387

Melted Form (sample):
  rating   type  duration_num
0  PG-13  Movie    108.853147
1      R  Movie    106.158103
2  TV-14  Movie    108.137765
3  TV-MA  Movie     95.243685

Proof: np.allclose(pivot.values, re_pivoted.values) == True (Lossless round trip verified)


### 10. Splitting, Merging & `validate` Demonstration

In [10]:
# Split DataFrame into two frames sharing key 'show_id'
shows_meta = df[["show_id", "title", "type", "release_year"]].iloc[:5000].copy()
shows_details = df[["show_id", "rating", "duration_num", "country"]].iloc[1000:6000].copy()

m_inner = pd.merge(shows_meta, shows_details, on="show_id", how="inner")
m_left = pd.merge(shows_meta, shows_details, on="show_id", how="left")
m_outer = pd.merge(shows_meta, shows_details, on="show_id", how="outer")

print(f"Inner Merge Row Count: {len(m_inner):,} (intersection of keys)")
print(f"Left Merge Row Count : {len(m_left):,} (all left rows preserved)")
print(f"Outer Merge Row Count: {len(m_outer):,} (union of both key sets)")

# Demonstration of validate parameter
print("\nTesting validate='1:1' on non-unique (many-to-many) merge:")
dup_sample = pd.concat([shows_details.iloc[:5], shows_details.iloc[:5]])
try:
    pd.merge(shows_meta, dup_sample, on="show_id", validate="1:1")
except pd.errors.MergeError as e:
    print(f"Caught MergeError as expected: {e}")

Inner Merge Row Count: 4,000 (intersection of keys)
Left Merge Row Count : 5,000 (all left rows preserved)
Outer Merge Row Count: 6,000 (union of both key sets)

Testing validate='1:1' on non-unique (many-to-many) merge:
Caught MergeError as expected: Merge keys are not unique in right dataset; not a one-to-one merge


**Explanation of Merge Differences**:
- `inner`: Retains only keys present in both DataFrames (4,000 overlapping rows).
- `left`: Retains all 5,000 rows from the left frame, padding missing right attributes with NaN.
- `outer`: Retains all unique keys from both frames (6,000 total rows) across the entire key space.

### 11. Rolling Mean by Group & Within-Group Ranking

In [11]:
# Sort by date within each group
df = df.sort_values(by=["type", "date_added"]).reset_index(drop=True)

# 7-row rolling mean of numeric duration per group
df["rolling_mean_duration"] = df.groupby("type", observed=True)["duration_num"].transform(
    lambda s: s.rolling(window=7, min_periods=1).mean()
)

# Rank within group based on duration
df["rank_in_group"] = df.groupby("type", observed=True)["duration_num"].rank(
    method="dense", ascending=False
)

df[["type", "title", "date_added", "duration_num", "rolling_mean_duration", "rank_in_group"]].head(10)

,type,title,date_added,duration_num,rolling_mean_duration,rank_in_group
0,Movie,Frank and Cindy,"April 1, 2016",70,70.000000,130.0
1,Movie,Frank and Cindy,"April 1, 2016",102,86.000000,98.0
2,Movie,Iverson,"April 1, 2016",88,86.666667,112.0
3,Movie,Jeremy Scott: The People's Designer,"April 1, 2016",109,92.250000,91.0
4,Movie,Beyond All Boundaries,"April 1, 2017",98,93.400000,102.0
5,Movie,Born to Be Free,"April 1, 2017",74,90.166667,126.0
6,Movie,Celluloid Man,"April 1, 2017",156,99.571429,44.0
7,Movie,Couples Counseling,"April 1, 2017",88,102.142857,112.0
8,Movie,Dancing Quietly,"April 1, 2017",70,97.571429,130.0
9,Movie,Deuces,"April 1, 2017",88,97.571429,112.0


### 12. Suspicious Rows Analysis

In [12]:
# Identify 3 anomalous records in the dataset
suspicious = df[df["date_added"].isna() | (df["duration_num"].isna()) | (df["release_year"] < 1940)].head(3)
suspicious[["show_id", "title", "type", "release_year", "date_added", "duration"]]

,show_id,title,type,release_year,date_added,duration
4262,80116008,Little Baby Bum: Nursery Rhyme Friends,Movie,2016,NaN,60 min
5244,81030762,Pioneers: First Women Filmmakers*,TV Show,1925,"December 30, 2018",1 Season
6221,70204989,Gunslinger Girl,TV Show,2008,NaN,2 Seasons


**Analysis of the 3 Suspicious Rows**:
1. **Missing `date_added`**: Certain legacy catalog additions lack ingest timestamp records, making temporal cohort analysis impossible.
2. **Pre-modern `release_year` (e.g. 1925/1942)**: Titles like historical archival footage or vintage propaganda films with extreme temporal distances from Netflix's core digital release catalog.
3. **Missing / Unmatched Duration**: Rows where duration formatting was corrupted or misaligned with rating metadata during source database ETL.

### 13. Export Cleaned Data to CSV & Parquet with Benchmarks

In [13]:
import os

csv_out = "cleaned_netflix_titles.csv"
parquet_out = "cleaned_netflix_titles.parquet"

# Prepare frame for parquet export (convert category to str if required)
df_save = df.copy()
for c in df_save.select_dtypes(include=["category"]).columns:
    df_save[c] = df_save[c].astype(str)

# Save to CSV and Parquet
df_save.to_csv(csv_out, index=False)
df_save.to_parquet(parquet_out, index=False)

# Compare file sizes
csv_size = os.path.getsize(csv_out) / (1024**2)
parquet_size = os.path.getsize(parquet_out) / (1024**2)

# Compare read/load times
t0 = time.time()
pd.read_csv(csv_out)
csv_time = time.time() - t0

t0 = time.time()
pd.read_parquet(parquet_out)
parquet_time = time.time() - t0

print("=== File Format Comparison ===")
print(f"CSV File Size    : {csv_size:.2f} MB | Load Time: {csv_time:.4f} seconds")
print(f"Parquet File Size: {parquet_size:.2f} MB | Load Time: {parquet_time:.4f} seconds")
print(f"Parquet Compression Savings: {((csv_size - parquet_size) / csv_size) * 100:.1f}%")
print(f"Parquet Speedup Multiplier : {csv_time / parquet_time:.1f}x faster")

=== File Format Comparison ===
CSV File Size    : 2.68 MB | Load Time: 0.0615 seconds
Parquet File Size: 1.36 MB | Load Time: 0.0909 seconds
Parquet Compression Savings: 49.1%
Parquet Speedup Multiplier : 0.7x faster
